# Part 2. Structured Outputs
As we have seen in "Part 1: Prompt Engineering", typically LLMs output some sort of text, i.e. there is no structure in it.
In this section, we will explore different ways of and tools (**Pydantic** and **Instructor**) available for generating structured outputs in the form of **JSON** (JavaScript Object Notation) objects.
Don't worry if you don't know what a JSON object is - it will make sense once you've seen an example.

The main advantages of relying on structured outputs:
- generate (more) predictable responses;
- allow to build (more) robust AI systems;
- format the data so that it's ready for downstream tasks (e.g., passing an output resulting from one LLM call to an AI Agent).

In [ ]:
#### UNCOMMENT ONE OF THE FOLLOWING:

### OPTION 1 - Running ollama locally

# When running ollama locally on your computer use the following command 
# to check the exact model names: ! ollama list.

llama8b = "llama3.1:latest"
llama3b = "llama3.2:3b"

### The end of OPTION 1.

############################################################################

### OPTION 2 - TACC Analysis Portal (https://tap.tacc.utexas.edu):

# # Start our ollama server in the background to host our LLMs
# from ollama_utils import start_ollama_server, stop_ollama_server

# # start ollama server
# start_ollama_server()

# # model names
# llama8b = "llama3.1:8b"
# llama3b = "llama3.2:latest"

### The end of OPTION 2.

In [ ]:
import json
import textwrap
import instructor
from datetime import date
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Annotated, Literal, List

from IPython.display import JSON

In [ ]:
# Create an OpenAI client
openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

#### Example 1: Structured outputs with information about scientists.

Imagine you would like an LLM to output information about a scientist in a specific JSON format:
```
{
    "name": "<scientist's name>",
    "occupation": "<scientist's occupation>",
    "date_of_birth": "<scientist's data of birth>",
    "facts": [
        <Fact 1>,
        <Fact 2>,
        ...
        <Fact n>
    ] 
}
```

In [ ]:
# Let's define a data schema/model using Pydantic's BaseModel class.
# Inside our class we must provide field names and their types.

# Note: Pydantic has a Field object which allows us to provide more 
# configuration parameters such as additional information, metadata,
# and validation constraints.

class Scientist(BaseModel):
    name: str
    occupation: str
    date_of_birth: str
    facts: List[str] = Field(..., description="A list of facts about the scientist.")

In [ ]:
# We can see the json schema of the Pydantic's model created above
# by calling the model_json_schema method.

code_generation_json_schema = json.dumps(
    Scientist.model_json_schema(),
    indent=2
)
print(code_generation_json_schema)

In [ ]:
# Now let's move to Instructor.
# Wrap the OpenaAI client into the Instructor client.

instructor_client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

In [ ]:
# Important: The response will be of type Scientist! 

response = instructor_client.chat.completions.create(
    model=llama8b,
    messages=[{"role": "user", "content": "Tell me about Albert Einstein"}],
    response_model=Scientist, # provide Pydantic model here
    max_retries=3,
    temperature=0.7
)

print("Type: ", type(response))
print(response)

In [ ]:
# We can now access all the data generated by the LLM by providing an attribute name.

print("Name:", response.name)
print("Occupation", response.occupation)
print("Date of Birth:", response.date_of_birth)
print("*" * 100)

# Since facts are of type list, we can iterate over it
for i, f in enumerate(response.facts):
    print(f"Fact {i+1}:")
    print(f)

In [ ]:
# We can also transform (serialize) the data from an instance of Scientist
# to JSON format.

response_json = response.model_dump_json()

print(response_json)
print(type(response_json))

#### Example 2: Synthetic Data Generation
Let's imagine that we need to generate some records about users.

In [ ]:
# Wrap the OpenaAI client into the Instructor client
instructor_client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

In [ ]:
# Create a Pydantic model

class User(BaseModel):
    first_name: str = Field(..., description="user's first name")
    second_name: str = Field(..., description="user's second name")
    date_of_birth: date = Field(..., description="user's date of birth")
    address_number: str = Field(..., description="building number where user lives")
    street_name: str = Field(..., description="street where user lives")
    city_name: str = Field(..., description="city in which user lives")
    state: str = Field(..., description="state (within) the United States of America where user lives")
    zip_code: str = Field(..., description="zip code", min_length=5, max_length=5)
    credit_score: int = Field(..., description="user's credit score", ge=300, le=850)
    sex: Literal["TACC_Male", "TACC_Female", "TACC_Other"] = Field(..., description="user's sex")

In [ ]:
system_prompt = "You are a data generation system. Your task is to create content based on the specifications provided by the user."
user_prompt = "Create a random user's data matching the data schema provided."

In [ ]:
generated_user_data = instructor_client.chat.completions.create(
    model=llama3b,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_model=User,
    max_retries=3,
    temperature=1.0
)

print("Type: ", type(generated_user_data))
print(generated_user_data)

In [ ]:
generated_user_data_json = generated_user_data.model_dump_json(indent=2)
print(generated_user_data_json)
print(f"Type: {type(generated_user_data_json)}")

In [ ]:
# Let's generate more users
# Warning: There will be some cases when an LLM will fail to generate 
# a correct output.

users_list = []

for i in range(20):
    print(f"Iteration # {i+1}")
    
    try:
        generated_user = instructor_client.chat.completions.create(
            model=llama8b,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_model=User,
            max_retries=2,
            temperature=1.0
        )
        
        users_list.append(generated_user)
    
    except Exception as ex:
        print(f"Something went wrong at iteration # {i+1}")
        print(ex)

In [ ]:
users_list

In [ ]:
# stop_ollama_server()